In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

from claimlens.data import load_data


In [83]:
df_full = load_data()
df_sample = pd.read_json('../raw_data/abcd_sample.json')

In [84]:
df_full['flow'] = df_full['scenario'].str['flow']
df_full['subflow'] = df_full['scenario'].str['subflow']

In [85]:
def quick_claim_text(turns, n=3):
    """Rough version: first n customer messages joined. Temporary until the group's cleaned column is ready."""
    msgs = [t["text"] for t in turns if t["speaker"] == "customer"]   # customer turns only
    return " ".join(msgs[:n])                                         # first n, joined with spaces

df_full["claim_text"] = df_full["delexed"].apply(quick_claim_text)
df_full["subflow"] = df_full["scenario"].str["subflow"]

In [86]:
df_full

,convo_id,scenario,original,delexed,split,flow,subflow,claim_text
0,3592,"{'personal': {'customer_name': 'crystal minh',...","[[agent, Hi!], [agent, How can I help you?], [...","[{'speaker': 'agent', 'text': 'hi!', 'turn_cou...",train,product_defect,return_size,"hi! i need to return an item, can you help me ..."
1,9489,{'personal': {'customer_name': 'alessandro pho...,"[[agent, good afternoon, how can I help you?],...","[{'speaker': 'agent', 'text': 'good afternoon,...",train,product_defect,refund_status,just wanted to check on the status of a refund...
2,3695,"{'personal': {'customer_name': 'joyce wu', 'me...","[[customer, HEY HO!], [agent, good afternoon, ...","[{'speaker': 'customer', 'text': 'hey ho!', 't...",train,storewide_query,timing_4,hey ho! i've got a promo code and i want to kn...
3,5798,"{'personal': {'account_id': 'xjiw8ufp3b', 'cus...","[[agent, Welcome to AcmeBrands! How can I help...","[{'speaker': 'agent', 'text': 'welcome to acme...",train,shipping_issue,manage,"hello, i would like to change my shipping deat..."
4,3647,{'personal': {'customer_name': 'norman bouchar...,"[[agent, Hello, how can i help you], [customer...","[{'speaker': 'agent', 'text': 'hello, how can ...",train,storewide_query,pricing_3,hello. i have a really cool party coming up. a...
...,...,...,...,...,...,...,...,...
10037,1622,"{'personal': {'account_id': 'yhezyjiyot', 'cus...","[[customer, HI], [agent, Thank you for contact...","[{'speaker': 'customer', 'text': 'hi', 'turn_c...",test,order_issue,manage_upgrade,hi i want to upgrade an order i have with you ...
10038,3293,{'personal': {'customer_name': 'rodriguez domi...,"[[agent, How may I help you?], [customer, Hi, ...","[{'speaker': 'agent', 'text': 'how may i help ...",test,manage_account,status_credit_missing,"hi, i think i am missing a $40 credit from a s..."
10039,8492,"{'personal': {'account_id': 'zatwqpzkkq', 'cus...","[[customer, I would like an extension on my su...","[{'speaker': 'customer', 'text': 'i would like...",test,subscription_inquiry,manage_extension,i would like an extension on my subscription. ...
10040,189,"{'personal': {'customer_name': 'joyce wu', 'me...","[[agent, Hi! How may I help you?], [customer, ...","[{'speaker': 'agent', 'text': 'hi! how may i h...",test,storewide_query,pricing_1,hi! how much do you charge to gift wrap items?...


In [87]:
df_full["subflow"].value_counts()

subflow
recover_username        361
reset_2fa               351
recover_password        336
status                  268
credit_card             266
                       ... 
boots_how_1              27
shirt_other_4            27
shirt_other_3            27
jeans_how_4              22
status_delivery_date      3
Name: count, Length: 96, dtype: int64

In [88]:
df_full

,convo_id,scenario,original,delexed,split,flow,subflow,claim_text
0,3592,"{'personal': {'customer_name': 'crystal minh',...","[[agent, Hi!], [agent, How can I help you?], [...","[{'speaker': 'agent', 'text': 'hi!', 'turn_cou...",train,product_defect,return_size,"hi! i need to return an item, can you help me ..."
1,9489,{'personal': {'customer_name': 'alessandro pho...,"[[agent, good afternoon, how can I help you?],...","[{'speaker': 'agent', 'text': 'good afternoon,...",train,product_defect,refund_status,just wanted to check on the status of a refund...
2,3695,"{'personal': {'customer_name': 'joyce wu', 'me...","[[customer, HEY HO!], [agent, good afternoon, ...","[{'speaker': 'customer', 'text': 'hey ho!', 't...",train,storewide_query,timing_4,hey ho! i've got a promo code and i want to kn...
3,5798,"{'personal': {'account_id': 'xjiw8ufp3b', 'cus...","[[agent, Welcome to AcmeBrands! How can I help...","[{'speaker': 'agent', 'text': 'welcome to acme...",train,shipping_issue,manage,"hello, i would like to change my shipping deat..."
4,3647,{'personal': {'customer_name': 'norman bouchar...,"[[agent, Hello, how can i help you], [customer...","[{'speaker': 'agent', 'text': 'hello, how can ...",train,storewide_query,pricing_3,hello. i have a really cool party coming up. a...
...,...,...,...,...,...,...,...,...
10037,1622,"{'personal': {'account_id': 'yhezyjiyot', 'cus...","[[customer, HI], [agent, Thank you for contact...","[{'speaker': 'customer', 'text': 'hi', 'turn_c...",test,order_issue,manage_upgrade,hi i want to upgrade an order i have with you ...
10038,3293,{'personal': {'customer_name': 'rodriguez domi...,"[[agent, How may I help you?], [customer, Hi, ...","[{'speaker': 'agent', 'text': 'how may i help ...",test,manage_account,status_credit_missing,"hi, i think i am missing a $40 credit from a s..."
10039,8492,"{'personal': {'account_id': 'zatwqpzkkq', 'cus...","[[customer, I would like an extension on my su...","[{'speaker': 'customer', 'text': 'i would like...",test,subscription_inquiry,manage_extension,i would like an extension on my subscription. ...
10040,189,"{'personal': {'customer_name': 'joyce wu', 'me...","[[agent, Hi! How may I help you?], [customer, ...","[{'speaker': 'agent', 'text': 'hi! how may i h...",test,storewide_query,pricing_1,hi! how much do you charge to gift wrap items?...


In [90]:
subflow = "promo_code_out_of_date"

texts = df_full.loc[df_full["subflow"] == subflow, "claim_text"]

print(f"=== {subflow} ({len(texts)} claims) ===\n")
for i, text in enumerate(texts, start=1):
    print(f"{i}. {text}\n")

=== promo_code_out_of_date (112 claims) ===

1. the promo code doesn't work i got this code 8 days ago. rodriguez domingo

2. good day, i'm trying to use a promo code but it keeps saying invalid sure my name is joyce wu it's crazy because i just got the code 9 days ago

3. hi i have a question about an order i made ok its crystal minh order id is <order_id>

4. i am trying to use a promo code but i keep getting an invalid error? norman bouchard well, that answers my other question.  it is eight days old.

5. hi i got a promo code 8 days ago but it does not work when i try to use it it says it is invalid my name is rodriguez domingo

6. hi ive got a promo code but its showing as invalid rodriguez domingo im a guest

7. hi i have a question about my order i made an order and tried to apply a promo code but it didn't take it said it is invalid

8. hello!!! i am really annoyed with this site right now i need someone to help me as soon as possible

9. i trying to use my promo code but i am 

In [ ]:
REFUND_SUBFLOWS = {
    "refund_initiate",
    "return_size",
    "return_color",
    "return_stain",
    "manage_dispute_bill",
    "bad_price_competitor",
    "status_mystery_fee",
    "refund_update",
    "manage_cancel",
}